# Scriven Testcase
This is a testcase for evaporation.  
A Vapor Bubble is growing in a superheated liquid.
Initially $t = 0$ there is no vapor in the system.  
The simulation is started from an analytical solution at $t_0 > 0$.  
We can then compare the simulated interface position in time to the analytically predicted.  
See `10.1016/j.ijheatmasstransfer.2021.121233`  

Note the following:
* A deviation from the source is the considerably larger surface tension, to deal with capillary timestep restrictions.
* How to take care of boundaries at the outflows?
* Paraview seems to not read time data correcty. Thus time annotation is missing in videos.
* Grid absolute size ($50\mu m$) is too small, then we get crude (or straight up wrong/empty) cut-cell/level-set rules. => rescale lengths to mm.
* How to initialize the level-set?  
    *   Quadratic:  + exact geometric represantation    - zero gradient at (0,0,0) not good for Stokes-Extension
    *   SDF:        + non-vanishing gradients           - inexact geometric representation; not-converging (at least for low res/amr)

In [1]:
#r "..\..\binaries\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [2]:
string proj = "XNSFE-ScrivenTestcase-v2";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [3]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR3_P2_T0_MPI4*	03/10/2023 22:52:45	8e4f969b...
#1: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR2_P2_T0_MPI4	03/06/2023 13:03:31	37ef5d2a...
#2: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR1_P2_T1_MPI4	03/06/2023 13:03:23	a77b1d2a...
#3: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR1_P2_T0_MPI4	03/06/2023 13:03:14	63ccdfcb...
#4: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR0_P2_T1_MPI4	03/06/2023 13:03:06	54da7383...
#5: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR0_P2_T0_MPI4	03/06/2023 13:03:04	8b95d0cf...
#6: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR3_P2_T1_MPI4*	03/06/2023 13:03:57	32a7054c...
#7: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR3_P2_T0_MPI4*	03/06/2023 13:03:48	b142d13e...
#8: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR2_P2_T1_MPI4*	03/06/2023 13:03:40	4ac52604...


## Grid and Boundary/Initial Value configuration

Note, that in contrast to the 1-D Testcases, the interface radius is given in terms of $\alpha_A$ :  
* $R_I(t) = 2 \xi \sqrt{\alpha_A t} $

In [4]:
static class Constants{
    public static double scale = 1000.0; // scale from 1m = 1000mm
    // careful order of declaration matters!!!
    public static double L = 187.5*1e-6 * Math.Pow(scale, 1);
    public static double Xi = 4.063;

    // material parameters
    public static double rho_A = 958.4*Math.Pow(scale, -3);
    public static double rho_B = 0.597*Math.Pow(scale, -3);

    public static double mu_A = 2.8e-4*Math.Pow(scale, -1);
    public static double mu_B = 1.26e-5*Math.Pow(scale, -1);

    public static double Sigma = 0.059 / 50000.0; // very small surface tension so that capillary timestep is bigger...

    public static double dT = 1.25; // rescale temperature to range [0, 1]
    public static double c_A = dT * 4216.0*Math.Pow(scale, 2);
    public static double c_B = dT * 2030.0*Math.Pow(scale, 2);

    public static double k_A = dT * 0.679*Math.Pow(scale, 1);
    public static double k_B = dT * 0.025*Math.Pow(scale, 1);

    public static double hVap  = 2.258e6*Math.Pow(scale, 2);
    public static double T_sat = 0.0; //373.15;
    public static double T_wall = T_sat + 1.0; //+ 1.25;

    public static double alpha_A = k_A / (c_A*rho_A);
    public static double alpha_B = k_B / (c_B*rho_B);
    public static double eps = 1.0-rho_B/rho_A;

    public static double r0 = 50*1e-6*Math.Pow(scale, 1);
    public static double t0 = Math.Pow(0.5 * r0 / Xi, 2) / alpha_A; // start time, to avoid singular massflux at t=0
    public static double tE = Math.Pow(2 * r0 / (2 * Xi),2)/alpha_A - t0; //1.5*1e-3 - t0; // Endtime, until twice the initial radius is reached
    // capillary timestep , for finest res + highest degree, use this, for comparability?!, Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) => 0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [5]:
public static class BoundaryAndInitialValueFactory { 

    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
            stw.WriteLine("static class BoundaryAndInitialValues {");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfacePos(){");
            stw.WriteLine("         return (X,t) => 2 * " + Constants.Xi + " * Math.Sqrt(" + Constants.alpha_A + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static Func<double[], double, double> InterfaceVel(){");
            stw.WriteLine("         return (X,t) => " + Constants.Xi + " * " + Constants.alpha_A + " / Math.Sqrt(" + Constants.alpha_A + " * (t+" + Constants.t0 + "));");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double Phi(double[] X, double t){"); // initialize quadratically
            stw.WriteLine("         bool signed = true;");
            stw.WriteLine("         if(signed)");
            stw.WriteLine("             return InterfacePos()(X,t) - Math.Sqrt(X[0]*X[0] + X[1]*X[1] + X[2]*X[2]);");
            stw.WriteLine("         else");
            stw.WriteLine("             return Math.Pow(InterfacePos()(X,t),2) - (X[0]*X[0] + X[1]*X[1] + X[2]*X[2]);");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityA(double[] X, double t){");
            stw.WriteLine("         return " + Constants.eps + " * InterfaceVel()(X,t);");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityXA(double[] X, double t){");
            stw.WriteLine("         double r = Math.Max(X.L2Norm(), BLAS.MachineEps);");
            stw.WriteLine("         double V = X[0]/r * VelocityA(X,0.0);");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return V * Math.Pow("+Constants.r0+"/r,2);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return V - 2 * V/"+Constants.r0+" * (r - "+Constants.r0+");"); // again linear extrapolation
            stw.WriteLine("         }");
            stw.WriteLine("     }");           
            stw.WriteLine("     public static double VelocityYA(double[] X, double t){");
            stw.WriteLine("         double r = Math.Max(X.L2Norm(), BLAS.MachineEps);");
            stw.WriteLine("         double V = X[1]/r * VelocityA(X,0.0);");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return V * Math.Pow("+Constants.r0+"/r,2);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return V - 2 * V/"+Constants.r0+" * (r - "+Constants.r0+");"); // again linear extrapolation
            stw.WriteLine("         }");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double VelocityZA(double[] X, double t){");
            stw.WriteLine("         double r = Math.Max(X.L2Norm(), BLAS.MachineEps);");
            stw.WriteLine("         double V = X[2]/r * VelocityA(X,0.0);");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return V * Math.Pow("+Constants.r0+"/r,2);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return V - 2 * V/"+Constants.r0+" * (r - "+Constants.r0+");"); // again linear extrapolation
            stw.WriteLine("         }");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     static Func<double, double> TempInitial = r => {");
            stw.WriteLine("         return "+Constants.T_wall+"-2*Math.Pow("+Constants.Xi+",2.0)*(("+Constants.rho_B+"*("+Constants.hVap+"+("+Constants.c_A+"-"+Constants.c_B+")*("+Constants.T_wall+"-"+Constants.T_sat+")))/("+Constants.rho_A+"*"+Constants.c_A+"))*MathNet.Numerics.Integration.GaussLegendreRule.Integrate(z => Math.Exp(-Math.Pow("+Constants.Xi+",2.0)*(Math.Pow(1-z,-2.0)-2*"+Constants.eps+"*z-1)),1-"+Constants.r0+"/r,1, 15);");
            stw.WriteLine("     };");
            stw.WriteLine("     public static double TemperatureA(double[] X, double t){");
            stw.WriteLine("         double r = X.L2Norm();");
            stw.WriteLine("         double T_r = "+Constants.Xi+" / ("+Constants.k_A+" / ("+Constants.rho_B+" * "+Constants.hVap+")) * Math.Sqrt("+Constants.alpha_A+" / "+Constants.t0+");");
            stw.WriteLine("         if(r >= "+Constants.r0+"){");
            stw.WriteLine("             return TempInitial(r);");
            stw.WriteLine("         }else{");
            stw.WriteLine("             return "+Constants.T_sat+" - ("+Constants.r0+"-r) * T_r;");
            stw.WriteLine("         }");
            stw.WriteLine("     }");
            stw.WriteLine("     public static double TemperatureABoundary(double[] X, double t){");
            stw.WriteLine("         return " + Constants.T_wall + ";");
            stw.WriteLine("     }");
            stw.WriteLine("     ");
            stw.WriteLine("     public static double VelocityB(double[] X, double t){");
            stw.WriteLine("         return 0.0;");
            stw.WriteLine("     }");
            stw.WriteLine("}");
            return stw.ToString();
        }
    }

    static public Formula Get_VAX() {
        return new Formula("BoundaryAndInitialValues.VelocityXA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_VAY() {
        return new Formula("BoundaryAndInitialValues.VelocityYA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_VAZ() {
        return new Formula("BoundaryAndInitialValues.VelocityZA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA() {
        return new Formula("BoundaryAndInitialValues.TemperatureA", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_TempA_Boundary() {
        return new Formula("BoundaryAndInitialValues.TemperatureABoundary", true, AdditionalPrefixCode:GetPrefixCode());
    }
    static public Formula Get_Phi() {
        return new Formula("BoundaryAndInitialValues.Phi", true, AdditionalPrefixCode:GetPrefixCode());
    }
}

## Begin Postprocessing

In [6]:
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Quadrature;

In [7]:
var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination);
sessions = sessions.OrderBy(s => s.KeysAndQueries["id:AMR"]).ToList();
sessions

#0: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR0_P2_T1_MPI4	03/06/2023 13:03:06	54da7383...
#1: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR0_P2_T0_MPI4	03/06/2023 13:03:04	8b95d0cf...
#2: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR1_P2_T1_MPI4	03/06/2023 13:03:23	a77b1d2a...
#3: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR1_P2_T0_MPI4	03/06/2023 13:03:14	63ccdfcb...
#4: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR2_P2_T0_MPI4	03/06/2023 13:03:31	37ef5d2a...


group by timestepsize

In [8]:
var groupedsessions = sessions.GroupBy(s => s.KeysAndQueries["NoOfTimesteps"]);
groupedsessions

index,value
0,"[ XNSFE-ScrivenTestcase-v2 XNSFE-ScrivenTestcase-v2_H8_AMR0_P2_T1_MPI4 03/06/2023 13:03:06 54da7383..., XNSFE-ScrivenTestcase-v2 XNSFE-ScrivenTestcase-v2_H8_AMR1_P2_T1_MPI4 03/06/2023 13:03:23 a77b1d2a... ]"
1,"[ XNSFE-ScrivenTestcase-v2 XNSFE-ScrivenTestcase-v2_H8_AMR0_P2_T0_MPI4 03/06/2023 13:03:04 8b95d0cf..., XNSFE-ScrivenTestcase-v2 XNSFE-ScrivenTestcase-v2_H8_AMR1_P2_T0_MPI4 03/06/2023 13:03:14 63ccdfcb..., XNSFE-ScrivenTestcase-v2 XNSFE-ScrivenTestcase-v2_H8_AMR2_P2_T0_MPI4 03/06/2023 13:03:31 37ef5d2a... ]"


In [18]:
double[] times = GenericBlas.Linspace(0.0, Constants.tE + Constants.t0, 1000);
double[] yValues = new double[times.Length];
var PhiFuncF = BoundaryAndInitialValueFactory.Get_Phi();
Func<double, double> PhiFunc = t => {
    return PhiFuncF.Evaluate(new double[3], t);
};
for(int i = 0; i < times.Length; i++){
    yValues[i] = PhiFunc(times[i] - Constants.t0);
}
Plot2Ddata RefPos = new Plot2Ddata(times, yValues, "analytical Interface Position");
RefPos.dataGroups.First().Format = new PlotFormat("k-");

#### loading timesteps, and estimating radius depending on volume

In [20]:
public static Plot2Ddata GetRadius(IEnumerable<ISessionInfo> s, int samples = -1){
    Plot2Ddata plt = new Plot2Ddata();
    foreach(var _s in s){
        plt = plt.Merge(GetRadius(_s, samples));
    }
    return plt;
}

public static Plot2Ddata GetRadius(ISessionInfo s, int samples = -1){
    List<ISessionInfo> sess = new List<ISessionInfo>();
    sess.Add(s);
    loadsessions(s);

    void loadsessions(ISessionInfo _s){
        if(_s.RestartedFrom != Guid.Empty){
            Console.WriteLine("{0} restarted from {1}, loading as well ...", _s.ID, _s.RestartedFrom);
            ISessionInfo restart = _s.Database.Controller.DBDriver.LoadSession(_s.RestartedFrom, _s.Database);
            sess.Add(restart);
            loadsessions(restart);
        }
    }
    var ts = sess.SelectMany(_s => _s.Timesteps.Where(t => t.TimeStepNumber.Length == 1)); 
    ts = ts.OrderBy(t => t.PhysicalTime);
    Console.WriteLine("{0} has {1} timesteps in total", s.ID, ts.Count());

    int N = samples < 0 ? ts.Count() : Math.Min(ts.Count(), samples + 1);
    double[] times = new double[N + 1];
    double[] yValues = new double[times.Length];

    times[0] = ts.First().PhysicalTime + Constants.t0;
    yValues[0] = GetRadius(ts.First());
    int ind = 0;
    double inc = (double)ts.Count() /  (double)N;
    for(int i = 1; i < N; i++){
        ind = (int)Math.Floor(i * inc);
        var _ts = ts.ElementAt(ind);
        times[i] = _ts.PhysicalTime + Constants.t0;
        yValues[i] = GetRadius(_ts);
    }
    times[N] = ts.Last().PhysicalTime + Constants.t0;
    yValues[N] = GetRadius(ts.Last());

    double GetRadius(ITimestepInfo ttt){
        var phi = (LevelSet)ttt.Fields.SingleOrDefault(f => f.Identification == "Phi");
        var trk = new LevelSetTracker((GridData)phi.GridDat, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new[] {"A", "B"}, phi);
        trk.UpdateTracker(0.0);
        var vol = BoSSS.Solution.XNSECommon.XNSEUtils.GetSpeciesArea(trk, trk.GetSpeciesId("B"));
        // vol = 4/3 * pi * r^3 * 1/8
        // r = (6 * vol / pi)^1/3
        double r = Math.Pow(6 * vol / Math.PI, 1.0/3.0);
        return r;
    }

    var splitname = s.Name.Split('_');
    string h = splitname[1];
    string p = splitname[3];
    string l = splitname[2];
    string t = splitname[4];
    Plot2Ddata Pos = new Plot2Ddata(times, yValues, h + ", " + l + ", " + p + ", " + t );
    return Pos;
}

In [21]:
Plot2Ddata plt = new Plot2Ddata();
foreach(var group in groupedsessions){
    plt = plt.Merge(GetRadius(group, 25));
}
// plt = plt.Merge(GetRadius(groupedsessions.ElementAt(0).First(), 25));
plt.ModFormat();
plt = plt.Merge(RefPos);
plt.ModLineColor("analytical", LineColors.Green);
plt.Xlabel = "Interface Position [mm]";
plt.Xlabel = "Time [s]";
plt.LegendAlignment = new[] {"i", "t", "l"};
plt.PlotNow()

54da7383-8f2e-4c45-83a4-a4968ffd4343 has 248 timesteps in total
a77b1d2a-3957-4059-bc23-e0aa228d57ad has 248 timesteps in total
8b95d0cf-4dbb-4c08-9964-26f47b89745c has 125 timesteps in total
63ccdfcb-59b3-4026-86e2-e731f8abaf32 has 125 timesteps in total
37ef5d2a-f9d3-4323-a2f0-3cdbec27350d has 125 timesteps in total
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 0.04 
 
 
 
 
 0.05 
 
 
 
 
 0.06 
 
 
 
 
 0.07 
 
 
 
 
 0.08 
 
 
 
 
 0.09 
 
 
 
 
 0.1 
 
 
 
 
 0 
 
 
 
 
 0.0001 
 
 
 
 
 0.0002 
 
 
 
 
 0.0003 
 
 
 
 
 0.0004 
 
 
 
 
 0.0005 
 
 
 
 
 0.0006 
 
 
 
 
 0.0007 
 
 
 
 
 0.0008 
 
 
 
 
 0.0009 
 
 
 
 
 0.001 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Time [s] 
 
 
 
 
 H8, AMR0, P2, T1 
 
 
 H8, AMR0, P2, T1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 H8, AMR1, P2, T1 
 
 
 H8, AMR1, P2, T1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 H8, AMR0, P2, T0 
 
 
 H8, AMR0, P2, T0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 H8, AMR1, P2, T0 
 
 
 H8, AMR1, P2, T0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 H8, AMR2, P2, T0 
 
 
 H8, AMR2, P2, T0 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 analytical Interface Position 
 
 
 analytical Interface Position 
 
 
 
	<path stroke='rgb( 0, 255, 0)' d='M81.6,177.1 L135.0,177.1 M62.2,542.4 L62.8,526.4 L63.4,519.7 L64.1,514.7 L64.7,510.4 L65.3,506.6
 L65.9,503.2 L66.5,500.0 L67.2,497.1 L67.8,494.3 L68.4,491.7 L69.0,489.3 L69.6,486.9 L70.3,484.6
 L70.9,482.5 L71.5,480.4 L72.1,478.3 L72.7,476.4 L73.4,474.4 L74.0,472.6 L74.6,470.8 L75.2,469.0
 L75.9,467.3 L76.5,465.6 L77.1,463.9 L77.7,462.3 L78.3,460.7 L79.0,459.2 L79.6,457.6 L80.2,456.1
 L80.8,454.7 L81.4,453.2 L82.1,451.8 L82.7,450.4 L83.3,449.0 L83.9,447.6 L84.5,446.3 L85.2,445.0
 L85.8,443.7 L86.4,442.4 L87.0,441.1 L87.6,439.8 L88.3,438.6 L88.9,437.4 L89.5,436.1 L90.1,434.9
 L90.7,433.8 L91.4,432.6 L92.0,431.4 L92.6,430.3 L93.2,429.1 L93.8,428.0 L94.5,426.9 L95.1,425.8
 L95.7,424.7 L96.3,423.6 L97.0,422.5 L97.6,421.5 L98.2,420.4 L98.8,419.4 L99.4,418.3 L100.1,417.3
 L100.7,416.3 L101.3,415.3 L101.9,414.3 L102.5,413.3 L103.2,412.3 L103.8,411.3 L104.4,410.3 L105.0,409.3
 L105.6,408.4 L106.3,407.4 L106.9,406.5 L107.5,405.5 L108.1,404.6 L108.7,403.7 L109.4,402.8 L110.0,401.8
 L110.6,400.9 L111.2,400.0 L111.8,399.1 L112.5,398.2 L113.1,397.3 L113.7,396.5 L114.3,395.6 L114.9,394.7
 L115.6,393.8 L116.2,393.0 L116.8,392.1 L117.4,391.3 L118.1,390.4 L118.7,389.6 L119.3,388.8 L119.9,387.9
 L120.5,387.1 L121.2,386.3 L121.8,385.5 L122.4,384.6 L123.0,383.8 L123.6,383.0 L124.3,382.2 L124.9,381.4
 L125.5,380.6 L126.1,379.8 L126.7,379.0 L127.4,378.3 L128.0,377.5 L128.6,376.7 L129.2,375.9 L129.8,375.2
 L130.5,374.4 L131.1,373.6 L131.7,372.9 L132.3,372.1 L132.9,371.4 L133.6,370.6 L134.2,369.9 L134.8,369.1
 L135.4,368.4 L136.0,367.7 L136.7,366.9 L137.3,366.2 L137.9,365.5 L138.5,364.7 L139.1,364.0 L139.8,363.3
 L140.4,362.6 L141.0,361.9 L141.6,361.2 L142.3,360.5 L142.9,359.8 L143.5,359.1 L144.1,358.4 L144.7,357.7
 L145.4,357.0 L146.0,356.3 L146.6,355.6 L147.2,354.9 L147.8,354.2 L148.5,353.5 L149.1,352.9 L149.7,352.2
 L150.3,351.5 L150.9,350.8 L151.6,350.2 L152.2,349.5 L152.8,348.8 L153.4,348.2 L154.0,347.5 L154.7,346.9
 L155.3,346.2 L155.9,345.6 L156.5,344.9 L157.1,344.3 L157.8,343.6 L158.4,343.0 L159.0,342.3 L159.6,341.7
 L160.2,341.0 L160.9,340.4 L161.5,339.8 L162.1,339.1 L162.7,338.5 L163.4,337.9 L164.0,337.3 L164.6,336.6
 L165.2,336.0 L165.8,335.4 L166.5,334.8 L167.1,334.2 L167.7,333.5 L168.3,332.9 L168.9,332.3 L169.6,331.7
 L170.2,331.1 L170.8,330.5 L171.4,329.9 L172.0,329.3 L172.7,328.7 L173.3,328.1 L173.9,327.5 L174.5,326.9
 L175.1,326.3 L175.8,325.7 L176.4,325.1 L177.0,324.5 L177.6,323.9 L178.2,323.3 L178.9,322.8 L179.5,322.2
 L180.1,321.6 L180.7,321.0 L181

In [22]:
var gp = plt.ToGnuplot();
gp.WriteScriptAndData("./Figures/gnuplot/data/Scriven_InterfacePos_p2"); // change a few settings by hand, see readme!

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


#### Export Sessions

In [4]:
sessions

#0: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR3_P2_T0_MPI4*	03/10/2023 22:52:45	8e4f969b...
#1: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR2_P2_T0_MPI4	03/06/2023 13:03:31	37ef5d2a...
#2: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR1_P2_T1_MPI4	03/06/2023 13:03:23	a77b1d2a...
#3: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR1_P2_T0_MPI4	03/06/2023 13:03:14	63ccdfcb...
#4: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR0_P2_T1_MPI4	03/06/2023 13:03:06	54da7383...
#5: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR0_P2_T0_MPI4	03/06/2023 13:03:04	8b95d0cf...
#6: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR3_P2_T1_MPI4*	03/06/2023 13:03:57	32a7054c...
#7: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR3_P2_T0_MPI4*	03/06/2023 13:03:48	b142d13e...
#8: XNSFE-ScrivenTestcase-v2	XNSFE-ScrivenTestcase-v2_H8_AMR2_P2_T1_MPI4*	03/06/2023 13:03:40	4ac52604...


In [5]:
var sess2plot = sessions.ElementAt(5);// pick session with finest grid and highest degree
string path2plot = Path.GetFullPath(Path.Combine("./Plots", sess2plot.Name));
sess2plot.Export().To(path2plot).WithSupersampling(2).WithShadowFields().Do(true);

Starting export process... Data will be written to the directory: d:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-Master\internal\src\private-rckmnn\notebooks\XNSFE-Paper\3D-Scriven\Plots\XNSFE-ScrivenTestcase-v2_H8_AMR0_P2_T0_MPI4


In [ ]:
#!pwsh
#!share --from csharp path2plot
pvpython ./Plots/scriven.py $path2plot

In [ ]:
display(new HtmlString(String.Format("<video width=\"1487\" height=\"1121\" controls><source src=\"{0}\" type=\'video/ogg; codecs=\"theora\"\'></video>",Path.Combine("./Plots", sess2plot.Name, "scriven.ogv") )))